# Proyecto RaSA — Entendimiento de datos
**Entregable 1 — Perfilamiento de datos**

Se exploran las cuatro fuentes compartidas por RaSA en la base `RaSaTransaccional` para entender su estructura, volumen y contenido, y dejar listas las observaciones que alimentan el análisis de calidad (Entregable 2) y las conclusiones de viabilidad (Entregable 3).

Fuentes:
- **F1. FuenteAreasDeServicio_Copia_E** — áreas de servicio por condado.
- **F2. FuenteTiposBeneficio_Copia_E** — tipos de beneficio.
- **F3. FuentePlanesBeneficio_Copia_E** — beneficios ofrecidos por cada plan (tabla central).
- **F4. FuenteCondicionesDePago_Copia_E** — condiciones de pago (referencia).

## 0. Configuración y utilidades

In [1]:
# Credenciales y conexión (use su usuario/clave del curso y su servidor de los tutoriales)
db_user = 'DB_202613_f_cucina'
db_psswd = '202425413'

# Base de datos de las fuentes
source_db_connection_string = 'jdbc:mysql://157.253.236.120:8080/RaSaTransaccional'  # AJUSTE host/puerto al suyo

# Driver JDBC de MySQL
path_jar_driver = '/workspaces/tareas-modelado-datos-2026-13/drivers/mysql-connector-j.jar' 

In [2]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder \
    .appName('Perfilamiento_RaSA') \
    .config('spark.jars', path_jar_driver) \
    .config('spark.driver.extraClassPath', path_jar_driver) \
    .config('spark.executor.extraClassPath', path_jar_driver) \
    .getOrCreate()

spark.conf.set('spark.sql.legacy.timeParserPolicy', 'LEGACY')

Picked up JAVA_TOOL_OPTIONS: -Xss512k -XX:+UseContainerSupport
Picked up JAVA_TOOL_OPTIONS: -Xss512k -XX:+UseContainerSupport
26/06/30 20:01:27 WARN Utils: Your hostname, codespaces-9ec919 resolves to a loopback address: 127.0.0.1; using 10.0.11.121 instead (on interface eth0)
26/06/30 20:01:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/06/30 20:01:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
def obtener_dataframe_de_bd(db_connection_string, sql, db_user, db_psswd):
    return spark.read.format('jdbc') \
        .option('url', db_connection_string) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()

def cargar_tabla(nombre):
    sql = f'(SELECT * FROM RaSaTransaccional.{nombre}) AS t'
    return obtener_dataframe_de_bd(source_db_connection_string, sql, db_user, db_psswd)

In [4]:
# --- Funciones de perfilamiento reutilizables ---
def resumen(df, nombre):
    print(f'=== {nombre} ===')
    print('Filas:', df.count(), '| Columnas:', len(df.columns))
    df.printSchema()

def nulos(df):
    total = df.count()
    fila = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).collect()[0]
    print(f'Total filas: {total}')
    for c in df.columns:
        n = fila[c]
        print(f'  {c}: {n} nulos ({100*n/total:.1f}%)')

def cardinalidad(df):
    fila = df.select([F.countDistinct(F.col(c)).alias(c) for c in df.columns]).collect()[0]
    for c in df.columns:
        print(f'  {c}: {fila[c]} valores distintos')

def frecuencias(df, columna, n=20):
    df.groupBy(columna).count().orderBy('count', ascending=False).show(n, truncate=False)

def anio(colname='Fecha'):
    # Extrae el año como texto (robusto aunque la fecha venga como string)
    return F.substring(F.col(colname).cast('string'), 1, 4)

In [5]:
# Verifique los nombres reales de las fuentes
tablas_bd = obtener_dataframe_de_bd(source_db_connection_string,
    "(SELECT table_name FROM information_schema.tables WHERE table_schema='RaSaTransaccional' AND table_name LIKE 'Fuente%') AS t",
    db_user, db_psswd)
tablas_bd.show(truncate=False)

TABLAS = {
    'areas':        'FuenteAreasDeServicio_Copia_E',
    'tipos':        'FuenteTiposBeneficio_Copia_E',
    'planes':       'FuentePlanesBeneficio_Copia_E',   # ajuste si en la BD es 'FuentesPlanes...'
    'condiciones':  'FuenteCondicionesDePago_Copia_E',
}

+-------------------------------+
|TABLE_NAME                     |
+-------------------------------+
|FuenteAreasDeServicio_Copia_E  |
|FuenteCondicionesDePago_Copia_E|
|FuentePlanesBeneficio_Copia_E  |
|FuenteTiposBeneficio_Copia_E   |
+-------------------------------+



## F1 — FuenteAreasDeServicio_Copia_E
**Significado de una fila:** un área de servicio definida sobre un condado (con su población, área y densidad). El negocio indica que deben ser **5409 áreas** y que cubren todos los condados del país.

In [6]:
areas = cargar_tabla(TABLAS['areas'])
resumen(areas, 'Áreas de servicio')

=== Áreas de servicio ===


Filas: 188815 | Columnas: 9
root
 |-- IdAreaDeServicio_T: integer (nullable = true)
 |-- NombreAreaDeServicio: string (nullable = true)
 |-- IdGeografia_T: integer (nullable = true)
 |-- Condado: string (nullable = true)
 |-- Estado: string (nullable = true)
 |-- PoblacionAct: double (nullable = true)
 |-- Area: double (nullable = true)
 |-- Densidad: double (nullable = true)
 |-- Fecha: integer (nullable = true)



Conteo de nulos por columna (completitud) y cardinalidad (¿es `idAreaServicio_T` única?).

In [7]:
print('--- Nulos ---'); nulos(areas)
print('--- Valores distintos ---'); cardinalidad(areas)

--- Nulos ---


Total filas: 188815
  IdAreaDeServicio_T: 6288 nulos (3.3%)
  NombreAreaDeServicio: 0 nulos (0.0%)
  IdGeografia_T: 6378 nulos (3.4%)
  Condado: 0 nulos (0.0%)
  Estado: 0 nulos (0.0%)
  PoblacionAct: 0 nulos (0.0%)
  Area: 2407 nulos (1.3%)
  Densidad: 2407 nulos (1.3%)
  Fecha: 0 nulos (0.0%)
--- Valores distintos ---


  IdAreaDeServicio_T: 5409 valores distintos
  NombreAreaDeServicio: 5412 valores distintos
  IdGeografia_T: 4251 valores distintos
  Condado: 1398 valores distintos
  Estado: 35 valores distintos
  PoblacionAct: 3729 valores distintos
  Area: 1957 valores distintos
  Densidad: 495 valores distintos
  Fecha: 3 valores distintos


Estadísticos de las columnas numéricas y revisión de las cifras del negocio.

In [11]:
areas.select('PoblacionAct', 'Area', 'Densidad').describe().show()

print('Áreas distintas (esperado 5409):', areas.select('IdGeografia_T').distinct().count())
print('Condados distintos:', areas.select('IdGeografia_T').distinct().count())
print('--- Estados ---'); frecuencias(areas, 'Estado')
print('--- Años (Fecha) ---'); areas.select(anio().alias('anio')).groupBy('anio').count().orderBy('anio').show()

+-------+-------------------+------------------+-----------------+
|summary|       PoblacionAct|              Area|         Densidad|
+-------+-------------------+------------------+-----------------+
|  count|             188815|            186408|           186408|
|   mean|4.147959051463602E7| 770.9518851122269|276.0979410754903|
| stddev|5.943835931334065E8|1263.4887340485252|851.3219978151402|
|    min|               82.0|          -24707.0|              0.0|
|    max|         4.72803E10|           88824.0|          14946.0|
+-------+-------------------+------------------+-----------------+



Áreas distintas (esperado 5409): 4252


Condados distintos: 4252
--- Estados ---


+--------------+-----+
|Estado        |count|
+--------------+-----+
|Indiana       |14150|
|Missouri      |13913|
|Georgia       |13822|
|Wisconsin     |12543|
|Michigan      |11598|
|Texas         |11261|
|South Dakota  |10930|
|Kansas        |10824|
|Kentucky      |10571|
|Oklahoma      |8183 |
|Virginia      |7553 |
|Florida       |7449 |
|Mississippi   |7234 |
|Illinois      |5965 |
|Montana       |5789 |
|North Carolina|5509 |
|New Jersey    |4900 |
|West Virginia |4613 |
|Ohio          |4033 |
|Pennsylvania  |3240 |
+--------------+-----+
only showing top 20 rows

--- Años (Fecha) ---


+----+-----+
|anio|count|
+----+-----+
|1800| 6282|
|2017|99211|
|2018|83322|
+----+-----+



**Observaciones — Áreas de servicio**

- **Volumen y unicidad:** la tabla tiene **188.815 filas** pero solo **5.409 `IdAreaDeServicio_T` distintos** — justo la cifra que reporta el negocio. Es decir, cada área aparece repetida ~35 veces: el identificador del área **no es único por fila** (fuerte duplicación). `NombreAreaDeServicio` tiene 5.412 distintos (3 más que ids), lo que sugiere pequeñas variaciones de nombre para una misma área.
- **Completitud:** la llave `IdAreaDeServicio_T` tiene **3,3% de nulos** (6.288 filas) e `IdGeografia_T` **3,4%** (6.378); `Area` y `Densidad` **1,3%** (2.407). Que la llave del área venga nula es crítico porque impide integrar esas filas.
- **Validez:** `Area` tiene un mínimo **negativo (−24.707)**, imposible para un área; `PoblacionAct` llega a **≈4,7×10¹⁰ (47 mil millones)** con una desviación enorme → valores atípicos/corruptos. `Densidad` mínimo 0.
- **Consistencia temporal:** `Fecha` toma 3 años: 2017 (99.211), 2018 (83.322) y **1800 (6.282)**, que es claramente inválido. Además **no aparece 2019**, aunque el negocio indica que la información va de 2017 a 2019.
- **Cobertura:** solo hay **35 estados** (de ~50) y 4.251 condados distintos.

## F2 — FuenteTiposBeneficio_Copia_E
**Significado de una fila:** un tipo de beneficio y sus atributos (si es EHB, si tiene límite cuantitativo, exclusiones, etc.). El negocio indica **170 tipos de beneficio**.

In [12]:
tipos = cargar_tabla(TABLAS['tipos'])
resumen(tipos, 'Tipos de beneficio')

=== Tipos de beneficio ===
Filas: 849 | Columnas: 9
root
 |-- IdTipoBeneficio_T: integer (nullable = true)
 |-- Nombre: string (nullable = true)
 |-- UnidadDelLimite: string (nullable = true)
 |-- EsEHB: string (nullable = true)
 |-- EstaCubiertaPorSeguro: string (nullable = true)
 |-- TieneLimiteCuantitativo: string (nullable = true)
 |-- ExcluidoDelDesembolsoMaximoDentroDeLaRed: string (nullable = true)
 |-- ExcluidoDelDesembolsoMaximoFueraDeLaRed: string (nullable = true)
 |-- Fecha: integer (nullable = true)



In [13]:
print('--- Nulos ---'); nulos(tipos)
print('--- Valores distintos ---'); cardinalidad(tipos)
print('Tipos distintos (esperado 170):', tipos.select('idTipoBeneficio_T').distinct().count())

--- Nulos ---


Total filas: 849
  IdTipoBeneficio_T: 0 nulos (0.0%)
  Nombre: 0 nulos (0.0%)
  UnidadDelLimite: 0 nulos (0.0%)
  EsEHB: 0 nulos (0.0%)
  EstaCubiertaPorSeguro: 0 nulos (0.0%)
  TieneLimiteCuantitativo: 0 nulos (0.0%)
  ExcluidoDelDesembolsoMaximoDentroDeLaRed: 0 nulos (0.0%)
  ExcluidoDelDesembolsoMaximoFueraDeLaRed: 0 nulos (0.0%)
  Fecha: 0 nulos (0.0%)
--- Valores distintos ---


  IdTipoBeneficio_T: 178 valores distintos
  Nombre: 178 valores distintos
  UnidadDelLimite: 63 valores distintos
  EsEHB: 3 valores distintos
  EstaCubiertaPorSeguro: 3 valores distintos
  TieneLimiteCuantitativo: 4 valores distintos
  ExcluidoDelDesembolsoMaximoDentroDeLaRed: 3 valores distintos
  ExcluidoDelDesembolsoMaximoFueraDeLaRed: 2 valores distintos
  Fecha: 2 valores distintos
Tipos distintos (esperado 170): 178


Distribución de las banderas (booleanas) y de la unidad del límite.

In [16]:
for c in ['EsEHB', 'EstaCubiertaPorSeguro', 'TieneLimiteCuantitativo',
          'ExcluidoDelDesembolsoMaximoDentroDeLaRed', 'ExcluidoDelDesembolsoMaximoFueraDeLaRed']:
    print(f'--- {c} ---'); frecuencias(tipos, c)
print('--- UnidadDelLimite ---'); frecuencias(tipos, 'UnidadDelLimite')

--- EsEHB ---


+-----+-----+
|EsEHB|count|
+-----+-----+
|Yes  |430  |
|No   |399  |
|True |20   |
+-----+-----+

--- EstaCubiertaPorSeguro ---


+---------------------+-----+
|EstaCubiertaPorSeguro|count|
+---------------------+-----+
|Yes                  |743  |
|No                   |98   |
|False                |8    |
+---------------------+-----+

--- TieneLimiteCuantitativo ---
+-----------------------+-----+
|TieneLimiteCuantitativo|count|
+-----------------------+-----+
|No                     |542  |
|Yes                    |271  |
|Nein                   |24   |
|Si                     |12   |
+-----------------------+-----+

--- ExcluidoDelDesembolsoMaximoDentroDeLaRed ---
+----------------------------------------+-----+
|ExcluidoDelDesembolsoMaximoDentroDeLaRed|count|
+----------------------------------------+-----+
|No                                      |787  |
|Yes                                     |60   |
|Algunas veces                           |2    |
+----------------------------------------+-----+

--- ExcluidoDelDesembolsoMaximoFueraDeLaRed ---


+---------------------------------------+-----+
|ExcluidoDelDesembolsoMaximoFueraDeLaRed|count|
+---------------------------------------+-----+
|No                                     |516  |
|Yes                                    |333  |
+---------------------------------------+-----+

--- UnidadDelLimite ---
+---------------------------------------------------------------------------------+-----+
|UnidadDelLimite                                                                  |count|
+---------------------------------------------------------------------------------+-----+
|                                                                                 |559  |
|Visit(s) per Year                                                                |64   |
|Visit(s) per Benefit Period                                                      |31   |
|Days per Year                                                                    |14   |
|Days per Benefit Period                                 

**Observaciones — Tipos de beneficio**

- **Completitud:** **0 nulos** en todas las columnas (buena completitud).
- **Unicidad:** **849 filas** con solo **178 `IdTipoBeneficio_T` distintos** → cada tipo se repite ~5 veces (no es único por fila). Además **178 ≠ 170** esperados: hay 8 tipos de más, posiblemente espurios o duplicados con variación.
- **Validez (banderas con codificación inconsistente y mezcla de idiomas):**
  - `EsEHB`: Yes (430) / No (399) y además **True (20)**.
  - `EstaCubiertaPorSeguro`: Yes (743) / No (98) y además **False (8)**.
  - `TieneLimiteCuantitativo`: No (542) / Yes (271) y además **Nein (24, alemán)** y **Si (12, español)** → 4 valores en 3 idiomas.
  - `ExcluidoDelDesembolsoMaximoDentroDeLaRed`: No (787) / Yes (60) y además **Algunas veces (2)**.
  - `UnidadDelLimite`: **559 vacíos** y cadenas corruptas/**triplicadas** (p. ej. "Visit(s) per YearVisit(s) per YearVisit(s) per Year").
- Estas banderas deben **estandarizarse a un único formato** (p. ej. Yes/No) antes de cualquier análisis.

## F3 — FuentePlanesBeneficio_Copia_E
**Significado de una fila:** un beneficio concreto ofrecido por un plan, en un área de servicio, con sus valores de copago/coseguro y condiciones. Es la **tabla central** que relaciona todas las demás. El negocio indica **301 planes en 2017 y 422 en 2018**, y para 2018 los máximos de copago/coseguro son **3300 y 100**.

In [17]:
planes = cargar_tabla(TABLAS['planes'])
resumen(planes, 'Planes-beneficio')

=== Planes-beneficio ===


Filas: 36036 | Columnas: 11
root
 |-- IdTipoBeneficio_T: long (nullable = true)
 |-- IdAreaDeServicio_T: long (nullable = true)
 |-- IdCondicionDePagoCopago_T: integer (nullable = true)
 |-- IdCondicionDePagoCoseguro_T: integer (nullable = true)
 |-- IdNivelServicio_T: integer (nullable = true)
 |-- IdPlan_T: string (nullable = true)
 |-- Fecha: string (nullable = true)
 |-- IdProveedor_T: integer (nullable = true)
 |-- valorCopago: integer (nullable = true)
 |-- valorCoseguro: integer (nullable = true)
 |-- cantidadLimite: integer (nullable = true)



In [18]:
print('--- Nulos ---'); nulos(planes)
print('--- Valores distintos ---'); cardinalidad(planes)

--- Nulos ---


Total filas: 36036
  IdTipoBeneficio_T: 2086 nulos (5.8%)
  IdAreaDeServicio_T: 2041 nulos (5.7%)
  IdCondicionDePagoCopago_T: 0 nulos (0.0%)
  IdCondicionDePagoCoseguro_T: 0 nulos (0.0%)
  IdNivelServicio_T: 0 nulos (0.0%)
  IdPlan_T: 0 nulos (0.0%)
  Fecha: 0 nulos (0.0%)
  IdProveedor_T: 0 nulos (0.0%)
  valorCopago: 0 nulos (0.0%)
  valorCoseguro: 0 nulos (0.0%)
  cantidadLimite: 30571 nulos (84.8%)
--- Valores distintos ---


  IdTipoBeneficio_T: 285 valores distintos
  IdAreaDeServicio_T: 6496 valores distintos
  IdCondicionDePagoCopago_T: 14 valores distintos
  IdCondicionDePagoCoseguro_T: 5 valores distintos
  IdNivelServicio_T: 3 valores distintos
  IdPlan_T: 393 valores distintos
  Fecha: 5 valores distintos
  IdProveedor_T: 125 valores distintos
  valorCopago: 49 valores distintos
  valorCoseguro: 25 valores distintos
  cantidadLimite: 41 valores distintos


Planes distintos por año (validación 301 en 2017, 422 en 2018) y estadísticos de los valores.

In [19]:
planes.withColumn('anio', anio()).groupBy('anio') \
      .agg(F.countDistinct('idPlan_T').alias('planes_distintos')) \
      .orderBy('anio').show()

planes.select('valorCopago', 'valorCoseguro', 'cantidadLimite').describe().show()

print('--- Máximos copago/coseguro 2018 (esperado 3300 y 100) ---')
planes.withColumn('anio', anio()).filter(F.col('anio') == '2018') \
      .select(F.max('valorCopago').alias('max_copago'),
              F.max('valorCoseguro').alias('max_coseguro')).show()

+----+----------------+
|anio|planes_distintos|
+----+----------------+
|1920|             212|
|2017|             203|
|2018|             286|
|Dec |             218|
+----+----------------+



+-------+------------------+------------------+-----------------+
|summary|       valorCopago|     valorCoseguro|   cantidadLimite|
+-------+------------------+------------------+-----------------+
|  count|             36036|             36036|             5465|
|   mean| 9.778416028416029| 24.29279054279054|129.3313815187557|
| stddev|100.41137967701272|36.959755494458506|951.2795862330698|
|    min|                 0|                 0|                1|
|    max|              3500|               100|            30000|
+-------+------------------+------------------+-----------------+

--- Máximos copago/coseguro 2018 (esperado 3300 y 100) ---


+----------+------------+
|max_copago|max_coseguro|
+----------+------------+
|      3500|         100|
+----------+------------+



**Observaciones — Planes-beneficio**

- **Completitud:** `IdTipoBeneficio_T` **5,8% nulos** (2.086) e `IdAreaDeServicio_T` **5,7%** (2.041) → llaves foráneas incompletas. `cantidadLimite` está **84,8% nula** (30.571): es esperable porque solo aplica a los tipos con límite cuantitativo; los valores no nulos van de **1 a 30.000 (sin ceros)**, lo cual es coherente con la regla del negocio.
- **Validez:** `valorCopago` llega a **3.500** y el máximo de **2018 es 3.500**, mientras el negocio indica un tope de **3.300** → excede en 200 (posible error/outlier). `valorCoseguro` máximo **100** (coincide con el negocio).
- **Conteo de planes:** hay **393 `IdPlan_T` distintos**, pero el conteo por año **no es confiable** porque `Fecha` viene en **formatos mezclados** (aparecen "1920", "Dec …" además de 2017/2018). Por eso no se obtienen los 301/422 esperados: primero hay que **estandarizar la fecha**.
- **Integridad referencial (consistencia):** los planes referencian **285** tipos de beneficio y **6.496** áreas, pero las fuentes respectivas solo tienen **178** tipos y **5.409** áreas → existen llaves foráneas que **apuntan a registros inexistentes**. `IdCondicionDePagoCopago_T` tiene 14 distintos (negocio: 15) y `IdCondicionDePagoCoseguro_T` 5 (coincide).

## F4 — FuenteCondicionesDePago_Copia_E
**Significado de una fila:** una condición de pago (referencia), de tipo Copago, Coseguro o Anticipado. El negocio indica **15 condiciones de copago y 5 de coseguro**.

In [20]:
condiciones = cargar_tabla(TABLAS['condiciones'])
resumen(condiciones, 'Condiciones de pago')

=== Condiciones de pago ===
Filas: 31 | Columnas: 3
root
 |-- IdCondicionesDePago_T: integer (nullable = true)
 |-- Descripcion: string (nullable = true)
 |-- Tipo: string (nullable = true)



In [21]:
print('--- Nulos ---'); nulos(condiciones)
print('--- Valores distintos ---'); cardinalidad(condiciones)
print('--- Tipo (esperado: 15 Copago, 5 Coseguro) ---'); frecuencias(condiciones, 'Tipo')
print('--- Descripcion ---'); frecuencias(condiciones, 'Descripcion')

--- Nulos ---
Total filas: 31
  IdCondicionesDePago_T: 0 nulos (0.0%)
  Descripcion: 0 nulos (0.0%)
  Tipo: 0 nulos (0.0%)
--- Valores distintos ---
  IdCondicionesDePago_T: 23 valores distintos
  Descripcion: 17 valores distintos
  Tipo: 6 valores distintos
--- Tipo (esperado: 15 Copago, 5 Coseguro) ---
+-------------+-----+
|Tipo         |count|
+-------------+-----+
|Copago       |19   |
|Coseguro     |7    |
|NaN          |2    |
|Copagado     |1    |
|Coseguridad  |1    |
|SinTipoCopago|1    |
+-------------+-----+

--- Descripcion ---
+--------------------------------+-----+
|Descripcion                     |count|
+--------------------------------+-----+
|Copay per Stay                  |3    |
|No Charge after deductible      |3    |
|No Charge                       |3    |
|Copay per Stay after deductible |3    |
|Coinsurance                     |3    |
|Coinsurance after deductible    |2    |
|Copay per Day with deductible   |2    |
|Copay                           |2    |
|N

**Observaciones — Condiciones de pago**

- **Completitud:** **0 nulos**.
- **Unicidad:** **31 filas** con solo **23 `IdCondicionesDePago_T` distintos** → el identificador **no es único** (hay duplicados); `Descripcion` también se repite (p. ej. "Copay per Stay" ×3, "No Charge" ×3).
- **Validez:** `Tipo` debería tomar solo Copago / Coseguro / Anticipado, pero aparecen **6 valores**: Copago (19), Coseguro (7) y los inválidos **NaN (2), Copagado (1), Coseguridad (1), SinTipoCopago (1)**. Los conteos (19/7) no cuadran con los **15/5** que indica el negocio, en parte por los duplicados y por los tipos mal escritos. Además **no aparece "Anticipado"**.

## Relaciones entre fuentes (insumo para el Entregable 3)
La tabla central `FuentePlanesBeneficio_Copia_E` integra las demás mediante estas llaves:

| Desde (Planes) | Hacia | Columna destino |
|---|---|---|
| idTipoBeneficio_T | FuenteTiposBeneficio_Copia_E | idTipoBeneficio_T |
| idAreaDeServicio_T | FuenteAreasDeServicio_Copia_E | idAreaServicio_T |
| idCondicionesDePagoCopago_T | FuenteCondicionesDePago_Copia_E | idCondicionesDePago_T (rol Copago) |
| idCondicionesDePagoCoseguro_T | FuenteCondicionesDePago_Copia_E | idCondicionesDePago_T (rol Coseguro) |

Además, `FuenteAreasDeServicio_Copia_E.idGeografía_T` identifica el condado (geografía). Estas relaciones se validan en el Entregable 3 (integridad referencial) y sustentan los análisis de cobertura, concentración y comportamiento de proveedores.

## Síntesis del perfilamiento (insumo para Calidad y Conclusiones)

| Fuente | Filas | Llave | ¿Llave única? | Hallazgos principales |
|---|---|---|---|---|
| Áreas de servicio | 188.815 | IdAreaDeServicio_T (5.409 dist.) | No (duplicación ~35x) | Nulos en la llave (3,3%); área negativa; población irreal; año 1800 inválido |
| Tipos de beneficio | 849 | IdTipoBeneficio_T (178 dist.) | No (repetición ~5x) | Banderas en varios idiomas (Nein/Si/True/False); unidades vacías/corruptas; 178≠170 |
| Planes-beneficio | 36.036 | IdPlan_T (393 dist.) | — | FKs nulas y a registros inexistentes (285 tipos, 6.496 áreas); fechas en formatos mezclados; copago 3.500>3.300 |
| Condiciones de pago | 31 | IdCondicionesDePago_T (23 dist.) | No (duplicados) | Tipo con valores inválidos (NaN, Copagado, Coseguridad); 19/7 ≠ 15/5 |

Estos hallazgos se analizan formalmente en el **Entregable 2** según las cuatro dimensiones de calidad (Completitud, Unicidad, Validez, Consistencia) y sustentan la **conclusión de viabilidad** del Entregable 3, donde se confirman las llaves de integración y se listan supuestos y dudas para el negocio.